<a href="https://colab.research.google.com/github/visiodepsp-lgtm/finadaptive/blob/main/KARUBET_AT_5_Ligas_1X2_CORREGIDO_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KARUBET · Escáner 1X2 de Apuesta Total — CORREGIDO

**Objetivo:** capturar **L / Empate / V** del listado de los cinco enlaces, como se ve en tu captura (sin abrir eventos ni mezclar mercados PA / SC). Guarda fecha, equipos y las tres cuotas verificadas, y acumula snapshots en **SQLite de Google Drive**.

**Qué corrige respecto a la versión anterior:** ya no busca botones `market-selection` ni exige encontrar títulos de `Pago Anticipado` o `Supercuota` dentro de la vista de detalle. Reconoce equipos de nuestro catálogo y asocia las **tres columnas iniciales de cuotas 1X2** del mismo renglón del fixture.

**Límite importante:** un sitio cargado en tu navegador puede presentar contenido distinto al navegador automático de Google Colab, incluso con `HTTP 200`. Si sigue mostrando 0, descarga el ZIP de diagnóstico (capturas `liga_XX.png`, texto, HTML) y compártemelo. No se inventan cuotas ni se rellenan con 1X2 de otros mercados. Las probabilidades implícitas sin margen que se guardan son **de mercado**, no probabilidades del modelo KARUBET.

Orden: instala, ejecuta configuración, lector 1X2, escaneo y descarga. Para continuidad, deja `GUARDAR_EN_DRIVE=True`.


In [6]:
# 1 · Preparar el navegador en Colab (solo la primera vez de cada sesión)
%pip -q install playwright pandas
!python -m playwright install --with-deps chromium

Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,164 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.6 MB]
Fetched 13.8 MB in 5s (2,656 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Buildin

In [7]:
# 2 · Configuración: pega aquí una URL de LIGA de Apuesta Total
# Ya incluimos el enlace que nos compartiste. Puedes añadir más ligas.
URLS_LIGAS = [
    'https://www.apuestatotal.com/apuestas-en-vivo/?fpath=/es-pe/spbkv3/sports/1/category?league=25',
    'https://www.apuestatotal.com/apuestas-en-vivo/?fpath=/es-pe/spbkv3/sports/1/category?league=74',
    'https://www.apuestatotal.com/apuestas-en-vivo/?fpath=/es-pe/spbkv3/sports/1/category?league=24',
    'https://www.apuestatotal.com/apuestas-en-vivo/?fpath=/es-pe/spbkv3/sports/1/category?league=38',
    'https://www.apuestatotal.com/apuestas-en-vivo/?fpath=/es-pe/spbkv3/sports/1/category?league=125',
]
MAX_EVENTOS_POR_LIGA = 40     # Prueba inicial: hasta 12 por liga; aumenta a 30 si funciona
ESPERA_MS = 8000              # Tiempo para cargar contenidos dinámicos
GUARDAR_EN_DRIVE = True      # Guarda SQLite en tu Google Drive si encuentra cuotas verificadas

# Catálogo de nombres aportado desde AT_NOMBRES.xlsx (no requiere subirlo).
# Incluye Villarreal. La lista ayuda a detectar pares de equipos.
CATALOGO_AT = [{'at': 'AC Milan', 'karubet': 'AC Milan'}, {'at': 'AS Roma', 'karubet': 'Roma'}, {'at': 'Atalanta', 'karubet': 'Atalanta'}, {'at': 'Bolonia', 'karubet': 'Bologna'}, {'at': 'Cagliari', 'karubet': 'Cagliari'}, {'at': 'Como', 'karubet': 'Como'}, {'at': 'Fiorentina', 'karubet': 'Fiorentina'}, {'at': 'Frosinone', 'karubet': 'Frosinone'}, {'at': 'Génova', 'karubet': 'Genoa'}, {'at': 'Inter de Milán', 'karubet': 'Inter'}, {'at': 'Juventus', 'karubet': 'Juventus'}, {'at': 'Lecce', 'karubet': 'Lecce'}, {'at': 'Monza', 'karubet': 'Monza'}, {'at': 'Nápoles', 'karubet': 'Napoli'}, {'at': 'Parma', 'karubet': 'Parma'}, {'at': 'Sassuolo', 'karubet': 'Sassuolo'}, {'at': 'SS Lazio', 'karubet': 'Lazio'}, {'at': 'Torino', 'karubet': 'Torino'}, {'at': 'Udinese', 'karubet': 'Udinese'}, {'at': 'Venezia', 'karubet': 'Venezia'}, {'at': 'Manchester City', 'karubet': 'Manchester City'}, {'at': 'Sunderland', 'karubet': 'Sunderland'}, {'at': 'Hull City', 'karubet': 'Hull City'}, {'at': 'Chelsea', 'karubet': 'Chelsea'}, {'at': 'Arsenal', 'karubet': 'Arsenal'}, {'at': 'Fulham', 'karubet': 'Fulham'}, {'at': 'Liverpool', 'karubet': 'Liverpool'}, {'at': 'Manchester United', 'karubet': 'Manchester United'}, {'at': 'Aston Villa', 'karubet': 'Aston Villa'}, {'at': 'Ipswich', 'karubet': 'Ipswich Town'}, {'at': 'Bournemouth', 'karubet': 'AFC Bournemouth'}, {'at': 'Crystal Palace', 'karubet': 'Crystal Palace'}, {'at': 'Tottenham Hotspur', 'karubet': 'Tottenham Hotspur'}, {'at': 'Newcastle', 'karubet': 'Newcastle United'}, {'at': 'Brentford', 'karubet': 'Brentford'}, {'at': 'Brighton', 'karubet': 'Brighton & Hove Albion'}, {'at': 'Coventry', 'karubet': 'Coventry City'}, {'at': 'Leeds', 'karubet': 'Leeds United'}, {'at': 'Everton', 'karubet': 'Everton'}, {'at': 'Nottingham Forest', 'karubet': 'Nottingham Forest'}, {'at': 'Getafe', 'karubet': 'Getafe'}, {'at': 'Dep La Coruña', 'karubet': 'Deportivo A Coruña'}, {'at': 'Sevilla', 'karubet': 'Sevilla'}, {'at': 'Racing Santander', 'karubet': 'Racing Santander'}, {'at': 'RCD Espanyol', 'karubet': 'Espanyol'}, {'at': 'Celta de Vigo', 'karubet': 'Celta de Vigo'}, {'at': 'Deportivo Alavés', 'karubet': 'Deportivo Alavés'}, {'at': 'Real Betis', 'karubet': 'Real Betis'}, {'at': 'Atlético de Madrid', 'karubet': 'Atlético de Madrid'}, {'at': 'Elche', 'karubet': 'Elche'}, {'at': 'Real Sociedad', 'karubet': 'Real Sociedad'}, {'at': 'Barcelona', 'karubet': 'FC Barcelona'}, {'at': 'CA Osasuna', 'karubet': 'Osasuna'}, {'at': 'Rayo Vallecano', 'karubet': 'Rayo Vallecano'}, {'at': 'Valencia', 'karubet': 'Valencia'}, {'at': 'Athletic Bilbao', 'karubet': 'Athletic Club'}, {'at': 'Málaga', 'karubet': 'Málaga'}, {'at': 'Real Madrid', 'karubet': 'Real Madrid'}, {'at': 'Levante', 'karubet': 'Levante'}, {'at': 'Auxerre', 'karubet': 'Auxerre'}, {'at': 'Mónaco', 'karubet': 'Monaco'}, {'at': 'Paris FC', 'karubet': 'Paris'}, {'at': 'Toulouse', 'karubet': 'Toulouse'}, {'at': 'Troyes', 'karubet': 'Troyes'}, {'at': 'Angers', 'karubet': 'Angers SCO'}, {'at': 'Lille', 'karubet': 'LOSC Lille'}, {'at': 'Le Mans', 'karubet': 'Le Mans'}, {'at': 'Stade Rennes', 'karubet': 'Rennes'}, {'at': 'Lyon', 'karubet': 'Olympique Lyonnais'}, {'at': 'Marsella', 'karubet': 'Olympique Marseille'}, {'at': 'Estrasburgo', 'karubet': 'Strasbourg'}, {'at': 'Lens', 'karubet': 'Lens'}, {'at': 'Le Havre', 'karubet': 'Le Havre'}, {'at': 'Lorient', 'karubet': 'Lorient'}, {'at': 'París Saint Germain', 'karubet': 'Paris Saint Germain'}, {'at': 'Niza', 'karubet': 'Nice'}, {'at': 'Brest', 'karubet': 'Brest'}, {'at': 'Friburgo', 'karubet': 'SC Freiburg'}, {'at': 'Bayer Leverkusen', 'karubet': 'Bayer 04 Leverkusen'}, {'at': 'Borussia Monchengladbach', 'karubet': 'Borussia Mönchengladbach'}, {'at': 'Hoffenheim', 'karubet': 'TSG Hoffenheim'}, {'at': 'Elversberg', 'karubet': 'Elversberg'}, {'at': 'Hamburgo', 'karubet': 'Hamburger SV'}, {'at': 'Union Berlin', 'karubet': 'FC Union Berlin'}, {'at': 'Mainz', 'karubet': 'FSV Mainz 05'}, {'at': 'Augsburgo', 'karubet': 'FC Augsburg'}, {'at': 'Borussia Dortmund', 'karubet': 'Borussia Dortmund'}, {'at': 'RB Leipzig', 'karubet': 'RB Leipzig'}, {'at': 'Eintracht Frankfurt', 'karubet': 'Eintracht Frankfurt'}, {'at': 'Bayern München', 'karubet': 'FC Bayern München'}, {'at': 'VfB Stuttgart', 'karubet': 'VfB Stuttgart'}, {'at': 'Schalke', 'karubet': 'Schalke 04'}, {'at': 'Paderborn', 'karubet': 'Paderborn'}, {'at': 'Colonia', 'karubet': 'FC Köln'}, {'at': 'Werder Bremen', 'karubet': 'Werder Bremen'}, {'at': 'Villarreal', 'karubet': 'Villarreal'}, {'at': 'Sabah FK', 'karubet': 'Sabah Masazir'}, {'at': 'Viking', 'karubet': 'Viking FK'}, {'at': 'Feyenoord', 'karubet': 'Feyenoord Rotterdam'}, {'at': 'Barcelona', 'karubet': 'FC Barcelona'}, {'at': 'Manchester City', 'karubet': 'Manchester City'}, {'at': 'Aston Villa', 'karubet': 'Aston Villa'}, {'at': 'París Saint Germain', 'karubet': 'Paris Saint Germain'}, {'at': 'Bayern München', 'karubet': 'FC Bayern München'}, {'at': 'Slavia Praga', 'karubet': 'Slavia Prague'}, {'at': 'Manchester United', 'karubet': 'Manchester United'}, {'at': 'RB Leipzig', 'karubet': 'RB Leipzig'}, {'at': 'AS Roma', 'karubet': 'Roma'}, {'at': 'Sporting Lisboa', 'karubet': 'Sporting CP'}, {'at': 'Nápoles', 'karubet': 'Napoli'}, {'at': 'Club Brugge', 'karubet': 'Club Brugge'}, {'at': 'AEK Atenas', 'karubet': 'AEK Athens'}, {'at': 'Galatasaray', 'karubet': 'Galatasaray Istanbul'}, {'at': 'Real Betis', 'karubet': 'Real Betis'}, {'at': 'Como', 'karubet': 'Como'}, {'at': 'Slovan Bratislava', 'karubet': 'SK Slovan Bratislava'}, {'at': 'Shakhtar Donetsk', 'karubet': 'FC Shakhtar Donetsk'}, {'at': 'Arsenal', 'karubet': 'Arsenal'}, {'at': 'Real Madrid', 'karubet': 'Real Madrid'}, {'at': 'Borussia Dortmund', 'karubet': 'Borussia Dortmund'}, {'at': 'Inter de Milán', 'karubet': 'Inter'}, {'at': 'Liverpool', 'karubet': 'Liverpool'}, {'at': 'FC Porto', 'karubet': 'FC Porto'}, {'at': 'PSV Eindhoven', 'karubet': 'PSV Eindhoven'}, {'at': 'Atlético de Madrid', 'karubet': 'Atlético de Madrid'}, {'at': 'VfB Stuttgart', 'karubet': 'VfB Stuttgart'}, {'at': 'Lille', 'karubet': 'LOSC Lille'}, {'at': 'Lens', 'karubet': 'Lens'}, {'at': 'FK Bodo Glimt', 'karubet': 'Bodoe/Glimt'}, {'at': 'LASK Linz', 'karubet': 'LASK Linz'}, {'at': 'Fenerbahce', 'karubet': 'Fenerbahce Istanbul'}, {'at': 'Villarreal', 'karubet': 'Villarreal'}]
NOMBRES_AT = ['AC Milan', 'AEK Atenas', 'AS Roma', 'Angers', 'Arsenal', 'Aston Villa', 'Atalanta', 'Athletic Bilbao', 'Atlético de Madrid', 'Augsburgo', 'Auxerre', 'Barcelona', 'Bayer Leverkusen', 'Bayern München', 'Bolonia', 'Borussia Dortmund', 'Borussia Monchengladbach', 'Bournemouth', 'Brentford', 'Brest', 'Brighton', 'CA Osasuna', 'Cagliari', 'Celta de Vigo', 'Chelsea', 'Club Brugge', 'Colonia', 'Como', 'Coventry', 'Crystal Palace', 'Dep La Coruña', 'Deportivo Alavés', 'Eintracht Frankfurt', 'Elche', 'Elversberg', 'Estrasburgo', 'Everton', 'FC Porto', 'FK Bodo Glimt', 'Fenerbahce', 'Feyenoord', 'Fiorentina', 'Friburgo', 'Frosinone', 'Fulham', 'Galatasaray', 'Getafe', 'Génova', 'Hamburgo', 'Hoffenheim', 'Hull City', 'Inter de Milán', 'Ipswich', 'Juventus', 'LASK Linz', 'Le Havre', 'Le Mans', 'Lecce', 'Leeds', 'Lens', 'Levante', 'Lille', 'Liverpool', 'Lorient', 'Lyon', 'Mainz', 'Manchester City', 'Manchester United', 'Marsella', 'Monza', 'Málaga', 'Mónaco', 'Newcastle', 'Niza', 'Nottingham Forest', 'Nápoles', 'PSV Eindhoven', 'Paderborn', 'Paris FC', 'Parma', 'París Saint Germain', 'RB Leipzig', 'RCD Espanyol', 'Racing Santander', 'Rayo Vallecano', 'Real Betis', 'Real Madrid', 'Real Sociedad', 'SS Lazio', 'Sabah FK', 'Sassuolo', 'Schalke', 'Sevilla', 'Shakhtar Donetsk', 'Slavia Praga', 'Slovan Bratislava', 'Sporting Lisboa', 'Stade Rennes', 'Sunderland', 'Torino', 'Tottenham Hotspur', 'Toulouse', 'Troyes', 'Udinese', 'Union Berlin', 'Valencia', 'Venezia', 'VfB Stuttgart', 'Viking', 'Villarreal', 'Werder Bremen']


In [8]:
# 3 · Lectura por POSICIÓN VISUAL del listado de cuotas 1X2 (no abre cada partido)
# No depende de etiquetas internas PA/SC ni de una clase CSS concreta.
EXTRACT_1X2_JS = r"""
({teamNames}) => {
  const norm = s => String(s||'').normalize('NFD').replace(/[\u0300-\u036f]/g,'').toLowerCase().replace(/[^a-z0-9]+/g,' ').replace(/\s+/g,' ').trim();
  const known = new Map();
  for (const n of teamNames||[]) known.set(norm(n), String(n));
  const rect = el => {
    try {const r=el.getBoundingClientRect();return {x:r.x,y:r.y,w:r.width,h:r.height};} catch {return null;}
  };
  const visible = el => {
    const r=rect(el); if (!r || r.w<2 || r.h<4) return false;
    if (r.x>window.innerWidth+300 || r.y>window.innerHeight+900 || r.x+r.w<0 || r.y+r.h<-300) return false;
    const s=getComputedStyle(el);
    return s.display!=='none' && s.visibility!=='hidden' && Number(s.opacity)!==0;
  };
  const nodes=[...document.querySelectorAll('body *')].slice(0,25000);
  const teams=[], prices=[];
  const seenT=new Set(),seenP=new Set();
  for(const el of nodes){
    if(!visible(el)) continue;
    const r=rect(el);
    if(r.w>520 || r.h>120) continue;  // sólo texto breve/etiquetas, no tarjetas enteras
    const txt=(el.innerText||el.textContent||'').trim();
    if(!txt || txt.length>65)continue;
    const n=norm(txt);
    const canonical=known.get(n);
    if(canonical){
      // Mantener el elemento más pequeño cuando varios padres repiten el mismo nombre.
      const area=r.w*r.h, key=n+'@'+Math.round(r.x/4)+'@'+Math.round(r.y/4);
      if(!seenT.has(key)){
        const childSame=[...el.children].some(c=>norm(c.innerText||c.textContent)===n && visible(c));
        if(!childSame){teams.push({name:canonical,key:n,el,x:r.x,y:r.y,w:r.w,h:r.h,area});seenT.add(key);}
      }
    }
    if(/^\d{1,3}[.,]\d{2,3}$/.test(txt)){
      const odd=Number(txt.replace(',','.'));
      if(odd<1.01||odd>999)continue;
      const childSame=[...el.children].some(c=>(c.innerText||c.textContent||'').trim()===txt && visible(c));
      if(childSame)continue;
      const key=Math.round(r.x/4)+'@'+Math.round(r.y/4);
      if(seenP.has(key))continue;
      const clickable=el.closest('button,[role="button"], [data-component="market-selection"]');
      const locked=Boolean(clickable&&(clickable.disabled || clickable.getAttribute('aria-disabled')==='true' ||
            /locked|suspended/i.test(String(clickable.className))));
      prices.push({odd,el,x:r.x,y:r.y,w:r.w,h:r.h,locked});seenP.add(key);
    }
  }
  teams.sort((a,b)=>a.y-b.y || a.x-b.x);
  prices.sort((a,b)=>a.y-b.y || a.x-b.x);
  const rows=[];const rowsKeys=new Set();
  for(let i=0;i<teams.length;i++){
    const a=teams[i];
    for(let j=i+1;j<teams.length && j<i+24;j++){
      const b=teams[j];
      if(b.key===a.key || b.y < a.y+4 || b.y-a.y>110 || Math.abs(b.x-a.x)>150)continue;
      let p=a.el;
      for(let depth=0;p && depth<13;depth++,p=p.parentElement){
        if(!p.contains(b.el)) continue;
        const r=rect(p); if(!r || r.w<400 || r.h>250 || r.h<52)continue;
        const cardPrices=prices.filter(q=>p.contains(q.el)&&q.y>=r.y-4&&q.y<=r.y+r.h+4 && !q.locked);
        if(cardPrices.length<3 || cardPrices.length>16) continue;
        // Evita captar un bloque con múltiples partidos.
        const homeCandidates=teams.filter(t=>p.contains(t.el) && t.y >=r.y-4 && t.y <r.y+r.h+4);
        const unique=new Set(homeCandidates.map(x=>x.key));
        if(unique.size>2)continue;
        const sorted=[...cardPrices].sort((x,y)=>x.x-y.x || x.y-y.y);
        // Las primeras tres columnas de cuotas son L / Empate / V.
        let trio=null;
        for(let k=0;k<=sorted.length-3;k++){
          const t=sorted.slice(k,k+3);
          if(t[0].x+15>=t[1].x || t[1].x+15>=t[2].x)continue;
          if(Math.max(...t.map(x=>x.y))-Math.min(...t.map(x=>x.y))>45)continue;
          const rowMid=(a.y+b.y)/2;
          if(Math.abs(t[0].y-rowMid)>90)continue;
          trio=t;break;
        }
        if(!trio)continue;
        // Un panel de 1X2 puede mostrar selección casa, empate, visita.
        const text=(p.innerText||p.textContent||'').replace(/\s+/g,' ').trim().slice(0,400);
        const midBox=trio[1].el.closest('button,[role="button"]');
        const midTxt=norm(midBox?.innerText || midBox?.textContent || '');
        const hasDraw=/\b(empate|draw|x)\b/.test(midTxt) || /\bempate\b/.test(norm(text));
        const key=a.key+'|'+b.key;
        const dateMatch=text.match(/\b(\d{1,2}\/\d{1,2})\s*[·\-.]?\s*(\d{1,2}:\d{2})\b/);
        const candidate={local:a.name,visitante:b.name,L:trio[0].odd,E:trio[1].odd,V:trio[2].odd,
          fecha_hora_texto:dateMatch?dateMatch[1]+' '+dateMatch[2]:'',
          doble_verificacion_empate:hasDraw,
          dom_fragment:text.slice(0,260),
          y:a.y, ancho:r.w, alto:r.h,
          pos_x:trio.map(v=>Math.round(v.x))};
        const prior=rows.find(x=>x.local===candidate.local&&x.visitante===candidate.visitante);
        if(!prior) rows.push(candidate);
        else if(!prior.doble_verificacion_empate&&hasDraw)Object.assign(prior,candidate);
        break;
      }
      if(rows.some(x=>x.local===a.name && x.visitante===b.name))break;
    }
  }
  rows.sort((a,b)=>a.y-b.y);
  return {rows,diagnostico:{title:document.title,url:location.href,body_chars:(document.body?.innerText||'').length,
    elementos:nodes.length,etiquetas_equipos:teams.length,etiquetas_cuotas:prices.length,
    body_inicio:(document.body?.innerText||'').slice(0,450).replace(/\n+/g,' | ')}};
}
"""


In [ ]:
# 4 · Escanear las cinco ligas: guardar cuotas de listado en CSV + SQLite persistente
import asyncio, json, sqlite3, re, unicodedata, time
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlparse, parse_qs
import pandas as pd
from IPython.display import display
from playwright.async_api import async_playwright

OUT = Path('/content/KARUBET_AT_1X2')
OUT.mkdir(parents=True,exist_ok=True)


def id_liga(url):
    m=re.search(r'league=(\d+)',str(url));return m.group(1) if m else 'desconocida'


def valid_url(url):
    try:
        u=urlparse(str(url).strip())
        return u.scheme=='https' and (u.hostname=='apuestatotal.com' or (u.hostname or '').endswith('.apuestatotal.com'))
    except (ValueError,TypeError):return False


def normaliza(s):
    s=unicodedata.normalize('NFD',str(s or ''))
    s=''.join(c for c in s if unicodedata.category(c)!='Mn').lower()
    return re.sub(r'[^a-z0-9]+',' ',s).strip()


def mercado_justo(l,e,v):
    # Probabilidad implicita de mercado sin margen
    inv=[1/l,1/e,1/v];m=sum(inv)
    return tuple(round(100*x/m,2) for x in inv)


async def escanear_liga(page,url,num):
    codigo=None;error='';ultimo={};filas={}
    try:
        respuesta=await page.goto(url,wait_until='domcontentloaded',timeout=55000)
        codigo=respuesta.status if respuesta else None
        await page.wait_for_timeout(ESPERA_MS)
        for paso in range(7):
            # Las cuotas pueden vivir dentro de un iframe.
            for f in page.frames:
                try:
                    data=await f.evaluate(EXTRACT_1X2_JS,{'teamNames':NOMBRES_AT})
                    if data.get('diagnostico',{}).get('body_chars',0)>ultimo.get('body_chars',0):
                        ultimo=data['diagnostico']
                    for row in data['rows']:
                        if not row.get('doble_verificacion_empate'):continue
                        key=(normaliza(row['local']),normaliza(row['visitante']),row.get('fecha_hora_texto',''))
                        filas[key]=row
                except Exception as exc:
                    error=str(exc)[:180]
            if filas and paso>=2:break
            await page.mouse.wheel(0,650)
            await page.wait_for_timeout(1900)
        # Captura diagnóstico visual en TODOS los casos, sin confundir HTTP 200 con contenido.
        await page.screenshot(path=str(OUT/f'liga_{num:02d}.png'),full_page=False,timeout=14000)
        if not filas:
            (OUT/f'liga_{num:02d}_html.html').write_text(await page.content(),encoding='utf-8')
            body=await page.locator('body').inner_text(timeout=5000)
            (OUT/f'liga_{num:02d}_texto.txt').write_text(body[:30000],encoding='utf-8')
        if filas:estado='CUOTAS_1X2_DETECTADAS'
        elif codigo in (403,429):estado='BLOQUEO_HTTP'
        elif any(w in str(ultimo).lower() for w in ['captcha','access denied','verify you are human','cloudflare']):estado='BLOQUEO_O_CAPTCHA'
        elif not ultimo.get('body_chars'):estado='PAGINA_SIN_CONTENIDO'
        else:estado='SIN_FILAS_1X2_EN_DOM'
        diag={'liga':id_liga(url),'url':url,'url_final':page.url,'http':codigo,'estado':estado,
              'eventos_con_1x2':len(filas),'etiquetas_equipos':ultimo.get('etiquetas_equipos'),
              'etiquetas_cuotas':ultimo.get('etiquetas_cuotas'),
              'longitud_texto':ultimo.get('body_chars'),'inicio_texto':ultimo.get('body_inicio',''),
              'error_js':error}
        return list(filas.values()),diag
    except Exception as exc:
        try:await page.screenshot(path=str(OUT/f'liga_{num:02d}.png'))
        except Exception:pass
        return [],{'liga':id_liga(url),'url':url,'http':codigo,'estado':'ERROR_NAVEGACION',
                   'eventos_con_1x2':0,'error_js':str(exc)[:300]}


async def run():
    run_time=datetime.now(timezone.utc).isoformat(timespec='seconds')
    records=[];diagnostics=[]
    urls=[u for u in URLS_LIGAS if valid_url(u)]
    async with async_playwright() as pw:
        browser=await pw.chromium.launch(headless=True,args=['--no-sandbox'])
        try:
            ctx=await browser.new_context(locale='es-PE',timezone_id='America/Lima',
                      viewport={'width':1500,'height':1000},device_scale_factor=1)
            for i,url in enumerate(urls,1):
                print(f'Liga {i}/{len(urls)}: {url}')
                p=await ctx.new_page()
                try:
                    rows,diag=await escanear_liga(p,url,i)
                    for row in rows[:MAX_EVENTOS_POR_LIGA]:
                        pr=mercado_justo(row['L'],row['E'],row['V'])
                        records.append({'capturado_utc':run_time,'liga':id_liga(url),
                          'url_liga':url,'fecha_hora_web':row.get('fecha_hora_texto',''),
                          'local_at':row['local'],'visitante_at':row['visitante'],
                          'L_1X2':row['L'],'E_1X2':row['E'],'V_1X2':row['V'],
                          'p_mercado_L_pct':pr[0],'p_mercado_E_pct':pr[1],'p_mercado_V_pct':pr[2],
                          'fuente':'1X2 visible en listado','estado':'TRES_CUOTAS_1X2'})
                    diagnostics.append(diag)
                    print('  HTTP',diag.get('http'),'·',diag['estado'],'·',len(rows),'partidos')
                finally:await p.close()
            await ctx.close()
        finally:await browser.close()
    return records,diagnostics

registros,diagnosticos=await run()
df=pd.DataFrame(registros)
dg=pd.DataFrame(diagnosticos)
display(dg[['liga','http','estado','eventos_con_1x2','etiquetas_equipos','etiquetas_cuotas']])
dg.to_csv(OUT/'diagnostico_ligas.csv',index=False,encoding='utf-8-sig')
if len(df):
    display(df[['liga','fecha_hora_web','local_at','visitante_at','L_1X2','E_1X2','V_1X2']])
    df.to_csv(OUT/'cuotas_1x2_actual.csv',index=False,encoding='utf-8-sig')
    if GUARDAR_EN_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        base=Path('/content/drive/MyDrive/KARUBET_Trading')
    else:base=OUT
    base.mkdir(parents=True,exist_ok=True)
    with sqlite3.connect(base/'cuotas_ligas_at.sqlite3') as con:
        con.execute('''CREATE TABLE IF NOT EXISTS cuotas_1x2_listado (
          capturado_utc TEXT NOT NULL,liga TEXT NOT NULL,url_liga TEXT NOT NULL,
          fecha_hora_web TEXT,local_at TEXT NOT NULL,visitante_at TEXT NOT NULL,
          l_1x2 REAL NOT NULL,e_1x2 REAL NOT NULL,v_1x2 REAL NOT NULL,
          p_mercado_l_pct REAL,p_mercado_e_pct REAL,p_mercado_v_pct REAL,
          fuente TEXT,estado TEXT,
          UNIQUE(capturado_utc,liga,fecha_hora_web,local_at,visitante_at))''')
        con.executemany('''INSERT OR IGNORE INTO cuotas_1x2_listado
          (capturado_utc,liga,url_liga,fecha_hora_web,local_at,visitante_at,
           l_1x2,e_1x2,v_1x2,p_mercado_l_pct,p_mercado_e_pct,p_mercado_v_pct,fuente,estado)
           VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?)''',
          [tuple(r[k] for k in ('capturado_utc','liga','url_liga','fecha_hora_web',
           'local_at','visitante_at','L_1X2','E_1X2','V_1X2','p_mercado_L_pct',
           'p_mercado_E_pct','p_mercado_V_pct','fuente','estado')) for r in registros])
        print('BD SQLite:',base/'cuotas_ligas_at.sqlite3','· filas totales:',
              con.execute('SELECT COUNT(*) FROM cuotas_1x2_listado').fetchone()[0])
else:
    print('NO hubo cuotas en el DOM de Colab. Abre liga_01.png y diagnostico_ligas.csv:')
    print('HTTP 200 por sí solo NO confirma que Colab vea tus partidos.')
print('Archivos de diagnóstico:',OUT)


Liga 1/5: https://www.apuestatotal.com/apuestas-en-vivo/?fpath=/es-pe/spbkv3/sports/1/category?league=25


In [ ]:
# 5 · Descargar paquete de diagnóstico y CSV (si existen)
from google.colab import files
from pathlib import Path
import zipfile
out=Path('/content/KARUBET_AT_1X2')
zip_path=Path('/content/KARUBET_AT_1X2_diagnostico.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in out.iterdir():
        if p.is_file():z.write(p,arcname=p.name)
print('Descargando:',zip_path)
files.download(str(zip_path))


## Cómo interpretar el resultado
- `CUOTAS_1X2_DETECTADAS`: se encontraron equipos y tres cuotas en el listado.
- `SIN_FILAS_1X2_EN_DOM`: Colab recibió texto de la página, pero no pudo asociar tres cuotas con dos equipos.
- `PAGINA_SIN_CONTENIDO`: el navegador automático no recibió el listado cargado.
- `BLOQUEO_O_CAPTCHA` / `BLOQUEO_HTTP`: la web no permite esa lectura desde Colab.

Los archivos `liga_XX.png` son **capturas realizadas por Colab**, no las de tu navegador. Si allí la web se ve vacía, el problema es de acceso/renderizado, no de reconocer 2.50 / 3.32 / 2.84. El notebook no pulsa apuestas ni realiza ninguna transacción.